In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot loss curves
axes[0].plot(train_losses, label='Train Loss', marker='o', markersize=3)
axes[0].plot(val_losses, label='Val Loss', marker='s', markersize=3)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Plot accuracy curves
axes[1].plot(train_accs, label='Train Accuracy', marker='o', markersize=3)
axes[1].plot(val_accs, label='Val Accuracy', marker='s', markersize=3)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'results' / 'figures' / 'training_curves.png', dpi=150)
plt.show()

print(f"✅ Training curves saved to results/figures/training_curves.png")

## Section 6: Training Curves Visualization

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc="Training")
    for batch_idx, (specs, labels) in enumerate(pbar):
        specs = specs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(specs)
        loss = criterion(outputs, labels)
        loss.backward()
        
        if config['training'].get('grad_clip', 0) > 0:
            nn.utils.clip_grad_norm_(model.parameters(), config['training']['grad_clip'])
        
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{100 * correct / total:.2f}%'
        })
    
    avg_loss = total_loss / len(train_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

def validate(model, val_loader, criterion, device):
    """Validate the model"""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for specs, labels in tqdm(val_loader, desc="Validating"):
            specs = specs.to(device)
            labels = labels.to(device)
            
            outputs = model(specs)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    avg_loss = total_loss / len(val_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

# Training loop
num_epochs = config['training']['num_epochs']
best_val_acc = 0.0
patience = config['training']['early_stopping']['patience']
patience_counter = 0

train_losses = []
val_losses = []
train_accs = []
val_accs = []

print(f"\n{'='*60}")
print(f"Starting training for {num_epochs} epochs")
print(f"{'='*60}\n")

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)
    
    scheduler.step()
    
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    print(f"  LR: {optimizer.param_groups[0]['lr']:.6f}")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'train_loss': train_loss,
            'val_loss': val_loss
        }
        torch.save(checkpoint, checkpoint_dir / 'best_model.pth')
        print(f"  ✅ Best model saved! (Val Acc: {val_acc:.2f}%)")
    else:
        patience_counter += 1
    
    # Early stopping
    if patience_counter >= patience:
        print(f"\n⚠️ Early stopping at epoch {epoch+1}")
        break

print(f"\n{'='*60}")
print(f"Training completed!")
print(f"Best validation accuracy: {best_val_acc:.2f}%")
print(f"{'='*60}")

## Section 5: Training Loop

In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss for handling class imbalance"""
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        p_t = torch.exp(-ce_loss)
        focal_loss = (1 - p_t) ** self.gamma * ce_loss
        
        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            focal_loss = alpha_t * focal_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

# Setup optimizer and loss
optimizer = optim.AdamW(
    model.parameters(),
    lr=config['training']['optimizer']['lr'],
    weight_decay=config['training']['optimizer']['weight_decay']
)

scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=config['training']['scheduler']['T_0'],
    T_mult=config['training']['scheduler']['T_mult'],
    eta_min=config['training']['scheduler']['eta_min']
)

criterion = FocalLoss(gamma=config['training']['loss']['gamma'])

# Setup directories
checkpoint_dir = PROJECT_ROOT / "results" / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

print(f"✅ Training configured:")
print(f"   Optimizer: AdamW (lr={config['training']['optimizer']['lr']})")
print(f"   Loss: Focal Loss (gamma={config['training']['loss']['gamma']})")
print(f"   Scheduler: CosineAnnealingWarmRestarts")
print(f"   Checkpoint dir: {checkpoint_dir}")

## Section 4: Setup Training

In [ ]:
class PatchEmbedding(nn.Module):
    """Patch embedding layer for spectrogram"""
    def __init__(self, input_shape, patch_size, embed_dim):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = (input_shape[0] // patch_size) * (input_shape[1] // patch_size)
        
        self.proj = nn.Conv2d(1, embed_dim, kernel_size=patch_size, stride=patch_size)
    
    def forward(self, x):
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        return x

class MultiHeadAttention(nn.Module):
    """Multi-head self-attention"""
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        attn = attn.softmax(dim=-1)
        attn = self.dropout(attn)
        
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        return self.dropout(x)

class TransformerBlock(nn.Module):
    """Transformer encoder block"""
    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadAttention(embed_dim, num_heads, dropout)
        
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_dim = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout)
        )
    
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class AudioSpectrogramTransformer(nn.Module):
    """Audio Spectrogram Transformer"""
    def __init__(self, config):
        super().__init__()
        self.patch_embed = PatchEmbedding(
            config['input_shape'],
            config['patch_size'],
            config['embed_dim']
        )
        
        num_patches = self.patch_embed.num_patches
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches, config['embed_dim']))
        
        self.blocks = nn.ModuleList([
            TransformerBlock(
                config['embed_dim'],
                config['num_heads'],
                config.get('mlp_ratio', 4.0),
                config.get('dropout', 0.1)
            )
            for _ in range(config['num_layers'])
        ])
        
        self.norm = nn.LayerNorm(config['embed_dim'])
        self.head = nn.Linear(config['embed_dim'], config['num_classes'])
        
        self._init_weights()
    
    def _init_weights(self):
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.apply(self._init_module_weights)
    
    def _init_module_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
    
    def forward(self, x):
        x = self.patch_embed(x)
        x = x + self.pos_embed
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        x = x.mean(dim=1)
        x = self.head(x)
        return x

# Build model
model_config = config['model'].copy()
model_config['input_shape'] = [config['preprocessing']['n_mels'], 1000]
model = AudioSpectrogramTransformer(model_config)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✅ Model built successfully!")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")

## Section 3: Build Model Architecture

In [ ]:
class AudioSpectrogramDataset(Dataset):
    """PyTorch Dataset for audio spectrograms"""
    def __init__(self, data_dir, transform=None):
        self.data_dir = Path(data_dir)
        self.files = sorted(self.data_dir.glob("*.npy"))
        self.transform = transform
        
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        # Load spectrogram
        spec = np.load(self.files[idx])
        
        # Apply transforms if any
        if self.transform:
            spec = self.transform(spec)
        
        # Extract label from filename (format: label_sampleid.npy)
        filename = self.files[idx].stem
        label = int(filename.split('_')[0])
        
        # Convert to tensor
        spec_tensor = torch.FloatTensor(spec).unsqueeze(0)  # Add channel dimension
        label_tensor = torch.LongTensor([label])[0]
        
        return spec_tensor, label_tensor

# Load datasets
print("Loading datasets...")
processed_dir = PROJECT_ROOT / "data" / "processed"

train_dataset = AudioSpectrogramDataset(processed_dir / "train")
val_dataset = AudioSpectrogramDataset(processed_dir / "val")
test_dataset = AudioSpectrogramDataset(processed_dir / "test")

print(f"✅ Datasets loaded:")
print(f"   Training samples: {len(train_dataset)}")
print(f"   Validation samples: {len(val_dataset)}")
print(f"   Test samples: {len(test_dataset)}")

# Create data loaders
batch_size = config['training']['batch_size']
num_workers = config['training']['num_workers']

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True
)

print(f"\n✅ Data loaders created:")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")
print(f"   Test batches: {len(test_loader)}")

## Section 2: Prepare Datasets

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import yaml
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import time
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Setup project path
PROJECT_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device}")
print(f"Project root: {PROJECT_ROOT}")

# Load configuration
config_path = PROJECT_ROOT / "configs" / "config.yaml"
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print(f"\n✅ Configuration loaded from {config_path}")

## Section 1: Import Libraries and Setup

# Audio Event Detection - Model Training & Analysis
## Audio Spectrogram Transformer for Emergency Sound Detection

This notebook provides a complete training pipeline for the AST model including:
1. Dataset preparation and loading
2. Model architecture implementation
3. Training configuration
4. Full training loop with validation
5. Checkpoint saving and early stopping
6. Training metrics visualization

In [ ]:
# Placeholder for 02_model_analysis.ipynb
{}